<a href="https://colab.research.google.com/github/atharva7471/AthoFolioo/blob/main/Day29-SklearnPipelines/29_titanic_using_pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier

In [3]:
df = pd.read_csv('train.csv')

In [4]:
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
739,740,0,3,"Nankoff, Mr. Minko",male,NaN,0,0,349218,7.8958,NaN,S
497,498,0,3,"Shellard, Mr. Frederick William",male,NaN,0,0,C.A. 6212,15.1000,NaN,S
262,263,0,1,"Taussig, Mr. Emil",male,52.0,1,1,110413,79.6500,E67,S
181,182,0,2,"Pernot, Mr. Rene",male,NaN,0,0,SC/PARIS 2131,15.0500,NaN,C
729,730,0,3,"Ilmakangas, Miss. Pieta Sofia",female,25.0,1,0,STON/O2. 3101271,7.9250,NaN,S


In [5]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [6]:
df.sample(2)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
541,0,3,female,9.0,4,2,31.2750,S
727,1,3,female,NaN,0,0,7.7375,Q


#Plan:

###So here we are going to make our pipeline in which it will be like--->
1) df ---> Using SI we will fill the 'Age' & 'Embarked' values
2) O/P of 1 (I/P) ---> Using OHE: 'Sex' & 'Embarked' transform values
3) O/P of 2 (I/P) ---> Using Scclar class MinMaxScale
4) O/P of 3 (I/P) ---> Feature Selection
5) O/P of 4 (I/P) ---> Using DecisionTree ---> Model

In [7]:
# Step1) Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['Survived']),
    df['Survived'],
    test_size = 0.2,
    random_state=42
)

In [8]:
X_train.sample(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
284,1,male,NaN,0,0,26.0,S
750,2,female,4.0,1,1,23.0,S


In [9]:
y_train.sample(2)

,Survived
571,1
355,0


In [10]:
#Create Imputation Transformers
ct1 = ColumnTransformer(
    [
        ('impute_age', SimpleImputer(), [2]),
        ('impute_embarked', SimpleImputer(strategy='most_frequent'),[6])
    ],remainder='passthrough'
)

In [11]:
# OHE Transformer
ct2 = ColumnTransformer(
    [
        ('ohe_sex_embarked', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [1,6])
    ], remainder='passthrough'
)

In [12]:
# Scaling
ct3 =ColumnTransformer(
    [
        ('scale', MinMaxScaler(), slice(0,10))
    ]
)

In [13]:
# Feature Selection
ct4 = SelectKBest(score_func=chi2, k=8)

In [14]:
# train object

ct5 = DecisionTreeClassifier()

##Now Create Pipeline

In [15]:
pipe = Pipeline([
    ('ct1', ct1),
    ('ct2', ct2),
    ('ct3', ct3),
    ('ct4', ct4),
    ('ct5', ct5)
])

###Pipeline() vs make_pipeline()
1) Pipeline(): Two parameters required name and steps <br>
2) make_pipeline(): Required only steps

Same Applied for ColumnTransformer and make_column_transformer

In [16]:
#Alternate:
#pipe = make_pipeline(ct1,ct2,ct3,ct4,ct5)

In [17]:
# train
pipe.fit(X_train, y_train)

Pipeline(steps=[('ct1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('ct2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('ct3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('ct4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x78c0f57bdd00>)),
                ('ct5', DecisionTreeClassifier())])

###Now you can explore Pipeline

In [18]:
# Step names
pipe.named_steps

{'ct1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'ct2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'ct3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'ct4': SelectKBest(k=8, score_func=<function chi2 at 0x78c0f57bdd00>),
 'ct5': DecisionTreeClassifier()}

In [19]:
pipe.named_steps['ct1']

ColumnTransformer(remainder='passthrough',
                  transformers=[('impute_age', SimpleImputer(), [2]),
                                ('impute_embarked',
                                 SimpleImputer(strategy='most_frequent'),
                                 [6])])

In [20]:
pipe.named_steps['ct1'].transformers_

[('impute_age', SimpleImputer(), [2]),
 ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6]),
 ('remainder',
  FunctionTransformer(accept_sparse=True, check_inverse=False,
                      feature_names_out='one-to-one'),
  [0, 1, 3, 4, 5])]

In [21]:
pipe.named_steps['ct1'].transformers_[0]

('impute_age', SimpleImputer(), [2])

In [22]:
pipe.named_steps['ct1'].transformers_[0][1]

SimpleImputer()

In [23]:
pipe.named_steps['ct1'].transformers_[0][1].statistics_

array([29.49884615])

In [24]:
pipe.named_steps['ct1'].transformers_[1][1].statistics_

array(['S'], dtype=object)

In [25]:
#Prediction

y_pred = pipe.predict(X_test)

In [26]:
y_pred

array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       0, 0, 0])

In [27]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.6256983240223464

###Cross validations using pipeline
Cross validations: itration of calculating accuracy score again and again

In [28]:
#Cross validations using cross_val_score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

np.float64(0.6391214419383433)

###Grid Search using pipeline

In [29]:
# gridsearchcv
params = {
    'ct5__max_depth': [1,2,3,4,5,None]
}


In [30]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('ct1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('ct2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          6])])),
                                       ('ct3',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('ct4',
                                        SelectKBest(k=8,
                                                    score_func=<function chi2 at 0x78c0f57bdd00>)),
                                       ('ct5', DecisionTreeClassifier())]),
             param_grid={'ct5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [31]:
grid.best_score_

np.float64(0.6391214419383433)

In [32]:
grid.best_params_

{'ct5__max_depth': 2}

###Export Pipeline
what if I want to use Pipeline on production level

In [33]:
import pickle as pkl
pkl.dump(pipe,open('pipe.pkl','wb'))